#instructions 
Neural Network SMS Text Classifier
You will be working on this project with Google Colaboratory.

After going to that link, create a copy of the notebook either in your own account or locally. Once you complete the project and it passes the test (included at that link), submit your project link below. If you are submitting a Google Colaboratory link, make sure to turn on link sharing for "anyone with the link."

We are still developing the interactive instructional content for the machine learning curriculum. For now, you can go through the video challenges in this certification. You may also have to seek out additional learning resources, similar to what you would do when working on a real-world project.

In this challenge, you need to create a machine learning model that will classify SMS messages as either "ham" or "spam". A "ham" message is a normal message sent by a friend. A "spam" message is an advertisement or a message sent by a company.

You should create a function called predict_message that takes a message string as an argument and returns a list. The first element in the list should be a number between zero and one that indicates the likeliness of "ham" (0) or "spam" (1). The second element in the list should be the word "ham" or "spam", depending on which is most likely.

For this challenge, you will use the SMS Spam Collection dataset. The dataset has already been grouped into train data and test data.

The first two cells import the libraries and data. The final cell tests your model and function. Add your code in between these cells.

In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# load data, preprocess, build & train model
from tensorflow.keras import layers

MAX_FEATURES = 5000
MAX_LEN = 100
EMBED_DIM = 32
BATCH_SIZE = 32
EPOCHS = 15

def load_tsv(path):
    """Load label\\tmessage TSV with no header. ham=0, spam=1 → sigmoid = P(spam)."""
    df = pd.read_csv(path, sep="\t", header=None, names=["label", "message"])
    df["label_id"] = df["label"].map({"ham": 0, "spam": 1}).astype("float32")
    return df

train_df = load_tsv(train_file_path)
valid_df = load_tsv(test_file_path)

print(train_df["label"].value_counts().to_string())
print(f"train={len(train_df)}, valid={len(valid_df)}")
train_df.head()


In [ ]:
# TextVectorization + embedding classifier
vectorize_layer = layers.TextVectorization(
    max_tokens=MAX_FEATURES,
    output_mode="int",
    output_sequence_length=MAX_LEN,
)
vectorize_layer.adapt(train_df["message"].values)

train_texts = train_df["message"].values
train_labels = train_df["label_id"].values
val_texts = valid_df["message"].values
val_labels = valid_df["label_id"].values

# string → TextVectorization → Embedding → GAP → Dense → P(spam)
# Vectorizer is part of the model so train and predict share the same path.
model = keras.Sequential(
    [
        layers.Input(shape=(), dtype=tf.string),
        vectorize_layer,
        layers.Embedding(MAX_FEATURES, EMBED_DIM),
        layers.GlobalAveragePooling1D(),
        layers.Dense(24, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid"),
    ]
)
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()

# Class weights: dataset is ~86% ham — without this the model may ignore spam
n_ham = int((train_labels == 0).sum())
n_spam = int((train_labels == 1).sum())
total = n_ham + n_spam
class_weight = {
    0: total / (2.0 * n_ham),
    1: total / (2.0 * n_spam),
}
print("class_weight:", class_weight)

history = model.fit(
    train_texts,
    train_labels,
    validation_data=(val_texts, val_labels),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    verbose=1,
)

val_loss, val_acc = model.evaluate(val_texts, val_labels, verbose=0)
print(f"Validation accuracy: {val_acc:.4f}  loss: {val_loss:.4f}")

# training curves
acc = history.history.get("accuracy", [])
val_acc_hist = history.history.get("val_accuracy", [])
loss = history.history.get("loss", [])
val_loss_hist = history.history.get("val_loss", [])
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="train acc")
plt.plot(epochs_range, val_acc_hist, label="val acc")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="train loss")
plt.plot(epochs_range, val_loss_hist, label="val loss")
plt.legend()
plt.title("Loss")
plt.tight_layout()
plt.show()


In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
    """
    Return [probability, label] for one SMS string.

    Probability is P(spam); label is 'spam' if P > 0.5 else 'ham'.
    """
    prob = float(model.predict(tf.constant([pred_text]), verbose=0)[0][0])
    label = "spam" if prob > 0.5 else "ham"
    return [prob, label]

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)


In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
